# pyceles Notebook: `CELES_MAIN.m` workflow replication

This notebook is a compact, interactive replication of the classic CELES
500-particle workflow using the current pyceles high-level API.

It demonstrates:

- automatic CuPy selection with a NumPy fallback;
- mixed-precision matrix-free GMRES;
- power-balance and cross-section diagnostics;
- near-field maps with a Poynting-vector overlay;
- far-field visualization and HDF5 persistence.


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / 'examples' / 'sphere_parameters.txt').is_file():
    _repo_root = _cwd
elif _cwd.name == 'notebooks' and (_cwd.parent / 'examples' / 'sphere_parameters.txt').is_file():
    _repo_root = _cwd.parent
else:
    raise FileNotFoundError(
        'Run this notebook from the repository root or its notebooks/ directory; '
        f'could not find examples/sphere_parameters.txt relative to {_cwd}.'
    )

sphere_parameters = _repo_root / 'examples' / 'sphere_parameters.txt'

# Prefer the checkout under test. This also lets the notebook run from a clone
# before pyceles has been installed into the active Python environment.
_src_path = _repo_root / 'src'
if str(_src_path) not in sys.path:
    sys.path.insert(0, str(_src_path))

import matplotlib.pyplot as plt
import numpy as np
import pyceles as pcl

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 180


def choose_backend() -> tuple[str, str, str | None]:
    try:
        import cupy as cp

        ndev = int(cp.cuda.runtime.getDeviceCount())
        if ndev > 0:
            name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode(errors='ignore')
            return 'cupy', 'inherit', name
    except Exception as exc:  # graceful fallback for no CuPy / no GPU / broken CUDA
        print(f'CuPy unavailable, falling back to NumPy: {exc}')
    return 'numpy', 'numpy', None


operator_backend, postprocessing_backend, gpu_name = choose_backend()
print(f'Operator backend: {operator_backend}')
print(f'Postprocessing backend: {postprocessing_backend}')
if gpu_name is not None:
    print(f'CuPy device: {gpu_name}')


## 1) Configure a 500-particle CELES-style experiment

The geometry comes from `examples/sphere_parameters.txt`. The setup keeps the
usual CELES wavelength, angular grids, and mixed-precision solve while exposing
backend and post-processing choices through `SimulationConfig`.


In [ ]:
sphere_data = np.loadtxt(sphere_parameters)
benchmark_n_particles = 500

radii = sphere_data[:benchmark_n_particles, 0].astype(float)
positions = sphere_data[:benchmark_n_particles, 1:4].astype(float)
refractive_indices = (
    sphere_data[:benchmark_n_particles, 4] + 1j * sphere_data[:benchmark_n_particles, 5]
).astype(np.complex128)
particles = pcl.spheres_from_arrays(
    positions=positions,
    radii=radii,
    refractive_indices=refractive_indices,
)

source = pcl.GaussianBeam(
    wavelength=550.0,
    medium_n=1.0 + 0j,
    polarization='TE',
    polar_angle=0.0,
    azimuthal_angle=0.0,
    beam_width=2000.0,
    focal_point=(0.0, 0.0, 0.0),
    amplitude=1.0,
)

cfg = pcl.SimulationConfig(
    wavelength=550.0,
    n_medium=1.0 + 0j,
    lmax=3,
    polar_angles=pcl.core.uniform_polar_grid(721),
    azimuthal_angles=pcl.core.uniform_periodic_azimuth_grid(360),
    solver_method='gmres',
    solver_rtol=1e-4,
    solver_restart=25,
    solver_maxiter=100,
    compute_dtype='complex64',
    accum_dtype='complex128',
    operator_backend=operator_backend,
    postprocessing_backend=postprocessing_backend,
    verbose=True,
)

sim = pcl.Simulation(cfg, particles=particles)


In [ ]:
run = sim.run(source)
solver = run.solver_result

if run.cross_sections is not None:
    cross_sections = run.cross_sections
    print(
        f'Cross sections: C_ext={cross_sections.extinction:.6e}, '
        f'C_sca={cross_sections.scattering:.6e}, '
        f'C_abs={cross_sections.local_absorption:.6e}'
    )


In [ ]:
nf_slice = pcl.compute_near_field_slice(
    run,
    axis_0_min=-4000.0,
    axis_0_max=4000.0,
    axis_1_min=-3000.0,
    axis_1_max=5000.0,
    dx=40.0,
    plane='y',
    plane_value=0.0,
    show_progress=True,
)

E, H = nf_slice.field_maps['total']
fig, _ = pcl.io.plot_nearfield_panels(
    nf_slice.axis_0,
    nf_slice.axis_1,
    E,
    H,
    particles=run.particles,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    real_limits=(-2.0, 2.0),
    abs_limits=(0.0, 2.0),
)
plt.show()

fig, _ = pcl.io.plot_nearfield_poynting_overlay(
    nf_slice.axis_0,
    nf_slice.axis_1,
    E,
    H,
    particles=run.particles,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    stride=5,
    quiver_color='w',
)
plt.show()


In [ ]:
ff = run.farfield
if ff.total is not None:
    spectrum = ff.total
else:
    spectrum = ff.scattered

Iff = pcl.io.far_field_intensity(spectrum)
fig, _ = pcl.io.plot_farfield_hemispheres(
    polar_angles=spectrum.beta,
    azimuthal_angles=spectrum.alpha,
    intensity=Iff,
    cmap='inferno',
    independent_scales=True,
)
plt.show()

out_h5 = _repo_root / 'outputs' / '01_pyceles_main_replication.h5'
out_h5.parent.mkdir(parents=True, exist_ok=True)
pcl.save_simulation_h5(run, nf_slice, out_h5)
print('Saved:', out_h5.resolve())
